# 09 — Model performance: population stability (PSI) + backtest feed

**Plain English:** This monitor watches *loans*; the sister
[mortgage-credit-risk-pd-lgd-ead](https://github.com/Jane511/mortgage-credit-risk-pd-lgd-ead)
project watches the *model* that scores them. Layer 4 of a five-layer monitoring
framework is **rating-system / model performance**. We add the link: the **PSI**
(population stability index) of the score distribution across vintages — has the
population the PD model was built on drifted? — and a **realised-default-by-grade**
table that is exactly the **backtest feed** the model needs (realised vs predicted).

**One-line terms**
- **PSI** — how far one distribution has shifted from a reference; <0.10 stable · 0.10–0.25 moderate · >0.25 significant.
- **Backtest** — comparing realised outcomes (this monitor) against the model's predicted PDs.
- **Grade** — a credit-score band, the model's risk ranking.

Predicted PDs live in the sister repo; here we compute PSI on a real panel risk
feature (credit score) and produce the realised side of the backtest, documenting
the intended linkage.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)

In [ ]:
# PSI of origination score & LVR across vintages (reference = calm 2015) --
orig = pd.concat([m.load_orig(v) for v in m.VINTAGES], ignore_index=True)
ref_v = "2015"
ref = orig[orig.vintage == ref_v]
rows = []
for feat in ["credit_score", "ltv"]:
    for v in [x for x in m.VINTAGES if x != ref_v]:
        val = m.psi(ref[feat], orig[orig.vintage == v][feat])
        rows.append({"feature": feat, "reference": ref_v, "vintage": v,
                     "PSI": round(val, 3), "classification": m.psi_class(val)})
psi_tbl = pd.DataFrame(rows)
psi_tbl.to_csv(m.OUT_TABLES / "09_psi.csv", index=False)
psi_tbl

In [ ]:
# Realised default by credit-score grade x vintage (the backtest feed) ----
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")
ever = (panel.assign(d=panel.stage == "Stage 3")
        .groupby(["vintage", "loan_seq"])
        .agg(d=("d", "max"), score=("credit_score", "first")).reset_index())
edges = [300, 620, 660, 700, 740, 780, 850]
labels = ["<620", "620-659", "660-699", "700-739", "740-779", "780+"]
ever["grade"] = pd.cut(ever.score, bins=edges, labels=labels, right=False)
grade = (ever.groupby(["grade", "vintage"], observed=False)["d"]
         .mean().mul(100).round(2).unstack("vintage"))
grade.to_csv(m.OUT_TABLES / "09_realised_default_by_grade.csv")
print("realised cumulative default rate (%) by score grade x vintage:")
grade